# Phase VI 

## Source
- Author: Santiago Sánchez
- Project: Aplicación de algoritmos de aprendizaje profundo a señales bioeléctricas para la identificación de segmentos de interés en el estudio de la epilepsia
- Year: 2025
- Repository: https://github.com/sansancas/TesisFinal
- Institution: Universidad del Valle de Guatemala

## Project Structure
The project consists in a system for automatic identification of epileptic seizures with EEG signals.  
   
├── api/          → FastAPI Server for production or commercial use of the model  
├── pt/           → PyTorch training pipeline  
├── tf/           → TensorFlow training pipeline  
├── ui/           → Tkinter desktop GUI  
└── .gitignore  

## Analysis

### pt/ - PyTorch pipeline
- Configuración global (utils.py)  
  
    Data parameters:  
    - Training, validation and evaluation paths
- Configuración global - utils.py  
    Data parameters:  
        - Training, validation and evaluation paths  
        - Maximum of registers  
        - Sampling strategy  
    - 

In [ ]:
from dataclasses import dataclass, dataclass, field
from pathlib import Path


@dataclass						# Class for storing configuration data with default values (Class definition)
class PipelineConfig:
	# Class atributes
	model: str = "hybrid"											# Model to use
	mode: str = "cv"												# Operation mode
	data_roots: list[Path] | None = field(default_factory=list)		# List of root pathd for data
	train_roots: list[Path] | None = field(default_factory=list)	# Paths for training data
    val_roots: list[Path] | None = field(default_factory=list)  	# Paths for validation data
	eval_roots: list[Path] | None = field(default_factory=list)  	# Paths for evaluation data
    records: list[Path] | None = field(default_factory=list)  		# List of specific records
    max_records: int = 40  											# Maximum total number of records to process
    max_records_train: int | None = None  							# Maximum for training
    max_records_val: int | None = None  							# Maximum for validation
    max_records_eval: int | None = None  							# Maximum for evaluation

    # Optional quotas per split for positive/negative records
    max_records_train_positive: int | None = None  					# Maximum positives in training
    max_records_train_negative: int | None = None  					# Maximum negatives in training
    max_records_val_positive: int | None = None  					# Maximum positives in validation
    max_records_val_negative: int | None = None  					# Maximum negatives in validation
    max_records_eval_positive: int | None = None  					# Maximum positives in evaluation
    max_records_eval_negative: int | None = None  					# Maximum negatives in evaluation
    max_per_patient: int = 1  										# Maximum records per patient
    include_background_only_records: bool = True  					# Include background-only records (no events)
    condition: str = "filtered"  									# Data filtering condition
    sampling_strategy: str = "none"  								# Sampling strategy: "none" by default
    sampling_seed: int | None = None  								# Seed for reproducible sampling
    undersample_target_positive_ratio: float | None = None  		# Target ratio for undersampling positives
    undersample_target_tolerance: float = 0.02  					# Tolerance for undersampling
    undersample_seed: int | None = None  							# Seed for undersampling
    montage: str = "ar"  											# Montage configuration (possibly for EEG)
    include_features: bool = False  								# Include additional features
    selected_features: list[str] = field(default_factory=list)  	# List of selected features
    time_step_labels: bool = False  								# Labels per time step
    batch_size: int = 8  											# Batch size for training
    epochs: int = 30  												# Number of training epochs
    folds: int = 5  												# Number of folds for cross-validation
    num_filters: int = 64  											# Number of filters in convolutional layers
    kernel_size: int = 7  											# Convolutional kernel size
    dropout: float = 0.3  											# Dropout rate for regularization
    rnn_units: int = 64  											# Units in RNN layers
    learning_rate: float = 1e-3  									# Learning rate
    optimizer: str = "adam"  										# Optimizer: "adam" by default
    optimizer_weight_decay: float = 0.0  							# Weight decay in optimizer
    optimizer_use_ema: bool = False  								# Use Exponential Moving Average in optimizer
    optimizer_ema_momentum: float = 0.99  							# Momentum for EMA
    jit_compile: bool = False  										# JIT (Just-In-Time) compilation
    window_sec: float = DEFAULT_WINDOW_SEC  						# Window in seconds (default constant)
    hop_sec: float = DEFAULT_HOP_SEC  								# Hop in seconds (default constant)
    epsilon: float = DEFAULT_EPS  									# Epsilon value (default constant)
    target_fs: float = DEFAULT_TARGET_FS  							# Target sampling frequency (default constant)
    max_training_minutes: float | None = None  						# Maximum training time in minutes
    preprocess_bandpass: bool | None = None  						# Apply bandpass filter in preprocessing
    preprocess_notch: bool | None = None  							# Apply notch filter
    preprocess_normalize: bool | None = None  						# Normalize data
    preprocess_n_harmonics: int | None = None  						# Number of harmonics in preprocessing
    use_class_weights: bool = True  								# Use class weights for balancing
    loss_type: str = "binary_crossentropy"  						# Loss function type (binary_crossentropy, focal, tversky)
    focal_alpha: float = 0.25  										# Alpha parameter for focal loss.
    focal_gamma: float = 2.0  										# Gamma parameter for focal loss.
    tversky_alpha: float = 0.7  									# Alpha parameter for Tversky loss.
    tversky_beta: float = 0.3  # Beta parameter for Tversky loss.
    tversky_gamma: float = 1.3333333333  # Gamma parameter for Tversky loss.
    patience: int = 5  # Patience for early stopping.
    min_lr: float = 1e-5  # Minimum learning rate.
    lr_schedule_type: str = "plateau"  # Learning rate scheduler type.
    cosine_annealing_period: int = 10  # Period for cosine annealing.
    cosine_annealing_min_lr: float | None = None  # Minimum LR for cosine annealing.
    save_metric_checkpoints: bool = True  # Save checkpoints based on metrics.
    seed: int = 42  # Seed for reproducibility.
    dry_run: bool = False  # Dry run mode without actual execution.
    output_dir: Path | None = None  # Output directory.
    epoch_time_log_path: Path | None = None  # Path for epoch time logs.
    checkpoint_dir: Path | None = None  # Checkpoint directory.
    verbose: int = 1  # Verbosity level (0-2).
    final_validation_split: float = 0.0  # Final validation split.
    use_tf_dataset: bool = False  # Use TensorFlow dataset.
    tf_data_shuffle_buffer: int | None = None  # Shuffle buffer for TF dataset.
    tf_data_prefetch: int | None = None  # Prefetch for TF dataset.
    tf_data_cache: str | bool | None = None  # Cache for TF dataset.
    write_tfrecords: bool = False  # Write TFRecords.
    tfrecord_dir: Path | None = None  # Directory for TFRecords.
    tfrecord_compression: str | None = None  # Compression for TFRecords.
    reuse_existing_tfrecords: bool = True  # Reuse existing TFRecords.
    dataset_cache_format: str = "npz"  # Dataset cache format.
    dataset_storage: str = "auto"  # Dataset storage.
    dataset_memmap_dir: Path | None = None  # Directory for memory mapping.
    dataset_auto_memmap_threshold_mb: float | None = 2048.0  # Threshold for auto memory mapping.
    feature_worker_processes: int | None = None  # Processes for feature extraction.
    feature_worker_chunk_size: int | None = 16  # Chunk size for workers.
    feature_parallel_min_windows: int = 32  # Minimum windows for parallelization.
    dataset_force_memmap_after_build: bool = False  # Force memory mapping after dataset build.
    transformer_embed_dim: int = 128  # Embedding dimension for transformer.
    transformer_num_layers: int = 4  # Number of layers in transformer.
    transformer_num_heads: int = 4  # Number of attention heads.
    transformer_mlp_dim: int = 256  # MLP dimension in transformer.
    transformer_dropout_rate: float = 0.1  # Dropout rate in transformer.
    transformer_use_se: bool = False  # Use Squeeze-and-Excitation in transformer.
    transformer_se_ratio: int = 16  # Ratio for SE.
    transformer_koopman_latent_dim: int = 0  # Latent dimension for Koopman.
    transformer_koopman_loss_weight: float = 0.0  # Loss weight for Koopman.
    transformer_use_reconstruction_head: bool = False  # Use reconstruction head.
    transformer_recon_weight: float = 0.0  # Reconstruction weight.
    transformer_recon_target: str = "signal"  # Reconstruction target.
    transformer_bottleneck_dim: int | None = None  # Bottleneck dimension.
    transformer_expand_dim: int | None = None  # Expand dimension.